# Cash Advance Underwriting: cash-flow scoring, limits and unit economics

A first-advance credit decision for a direct-to-consumer cash advance / earned-wage-access product, built the way a lender's credit team would build it: cash-flow features from linked bank data, an explainable champion scorecard against a machine-learning challenger, risk-based limits, loss curves, and a profit-maximizing cutoff.

> **Data note.** No public loan-level data exists for cash advance products, so the data is simulated from stated assumptions (`src/cashadvance/config.py`, `docs/assumptions.md`). The methods transfer directly to real bank-transaction data.

In [1]:
import sys, json
sys.path.insert(0, '../src')
import pandas as pd, numpy as np
from cashadvance.simulate import simulate
from cashadvance.features import build_features, FEATURES
from cashadvance.models import train_and_evaluate, decile_table
from cashadvance.strategy import compare_strategies, cutoff_curve
from cashadvance.config import SIM, PRODUCT
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)

## 1. Applicants and 90 days of bank activity

30,000 applicants across 12 monthly cohorts. Historical policy was a test-and-learn cell that approved every applicant at a flat $100, so outcomes are observed for everyone (no reject-inference problem).

In [2]:
t = simulate(SIM)
print({k: len(v) for k, v in t.items() if not k.startswith('_')})
t['deposits'].head()

{'members': 30000, 'deposits': 426677, 'balances': 30000, 'outcomes': 30000}


,member_id,day,amount,source
0,0,1,19.64,transfer
1,0,2,1023.81,payroll
2,0,16,1075.49,payroll
3,0,30,1041.51,payroll
4,0,40,88.99,transfer


## 2. Cash-flow features

Thin-file applicants are underwritten on what the bank account shows: recurring income, pay regularity, how close to zero the balance runs, and overdraft behaviour.

In [3]:
df = build_features(t['members'], t['deposits'], t['balances']).merge(t['outcomes'], on='member_id')
df[FEATURES].describe().T[['mean', '50%', 'min', 'max']].round(2)

,mean,50%,min,max
est_monthly_income,2883.44,2509.58,202.97,17170.01
paycheck_count,8.77,7.00,0.00,36.00
paycheck_gap_cv,0.20,0.00,0.00,2.00
detected_pay_period_days,11.80,14.00,1.00,83.00
days_since_last_paycheck,6.75,5.00,0.00,90.00
income_concentration,0.94,0.98,0.00,1.00
avg_daily_balance_to_income,0.40,0.37,-1.14,2.40
min_balance_to_income,-0.06,-0.03,-2.40,1.87
pct_days_negative,0.18,0.02,0.00,1.00
overdraft_events,1.67,1.00,0.00,11.00


## 3. Champion scorecard vs. XGBoost challenger

Development cohorts 1-8 (75/25 train/validation); cohorts 9-12 held out as an out-of-time test. The XGBoost challenger uses monotonic constraints so every feature moves risk in a defensible direction.

In [4]:
out = train_and_evaluate(df)
res = out['results']
rows = []
for m in ['champion_scorecard', 'challenger_xgboost']:
    for s in ['train', 'validation', 'out_of_time']:
        rows.append({'model': m, 'sample': s, **res[m][s]})
pd.DataFrame(rows)

,model,sample,auc,gini,ks,bad_rate,n
0,champion_scorecard,train,0.7265,0.4530,0.3412,0.0990,14797
1,champion_scorecard,validation,0.7163,0.4327,0.3458,0.0992,5063
2,champion_scorecard,out_of_time,0.7050,0.4100,0.3118,0.1092,10140
3,challenger_xgboost,train,0.7745,0.5491,0.4097,0.0990,14797
4,challenger_xgboost,validation,0.7267,0.4535,0.3474,0.0992,5063
5,challenger_xgboost,out_of_time,0.7146,0.4293,0.3282,0.1092,10140


In [5]:
pd.Series(res['information_value'], name='IV').to_frame()

,IV
ending_balance_to_income,0.469
avg_daily_balance_to_income,0.456
min_balance_to_income,0.417
pct_days_negative,0.353
est_monthly_income,0.316
balance_trend_to_income,0.310
income_concentration,0.183
overdraft_events,0.153
requested_to_income,0.151
balance_volatility_to_income,0.109


![ROC](../reports/figures/roc_out_of_time.png)

![Calibration](../reports/figures/calibration_deciles.png)

## 4. Adverse action reasons

Every decline maps to plain-language principal reasons, ranked by points lost against the best bin, as ECOA / Regulation B adverse action notices require.

In [6]:
champ = out['champion']; tr, va, oot = out['splits']
s = oot.head(6)
pd.DataFrame({'score': champ.score(s[FEATURES]).round(0),
              'reasons': champ.reason_codes(s[FEATURES]).apply('; '.join)})

,score,reasons
0,562.0,Low account balance relative to income; Bank a...
2,577.0,Low account balance relative to income; Income...
3,595.0,Low account balance relative to income; Income...
5,513.0,Low account balance relative to income; Income...
6,535.0,Low account balance relative to income; Income...
11,598.0,Income deposits too low; Low account balance r...


## 5. Cutoff, limits and unit economics

Decisions use the model; outcomes come from the simulated ground truth, the way a champion/challenger test cell would read out. Each approved member is followed for 12 months: membership fees, instant-delivery fees, funding cost, processing, servicing, losses and acquisition cost.

In [7]:
p = champ.predict_proba(oot[FEATURES])
truth = t['_truth'].set_index('member_id').loc[oot['member_id']].reset_index()
cc = cutoff_curve(p, truth)
best = cc.loc[cc['contribution_per_1000_applicants'].idxmax()]
cc[['approval_target', 'cutoff_pd', 'first_advance_co_rate', 'net_loss_rate_of_originations', 'contribution_per_member', 'contribution_per_1000_applicants', 'payback_month']]

,approval_target,cutoff_pd,first_advance_co_rate,net_loss_rate_of_originations,contribution_per_member,contribution_per_1000_applicants,payback_month
0,0.30,0.0522,0.0341,0.0116,54.17,16250.0,4
1,0.35,0.0590,0.0372,0.0133,48.54,16991.0,4
2,0.40,0.0658,0.0410,0.0146,44.47,17788.0,4
3,0.45,0.0730,0.0424,0.0158,40.84,18378.0,5
4,0.50,0.0802,0.0457,0.0171,36.91,18456.0,5
5,0.55,0.0886,0.0486,0.0183,33.53,18440.0,5
6,0.60,0.0984,0.0517,0.0197,29.34,17607.0,5
7,0.65,0.1087,0.0569,0.0211,25.31,16453.0,6
8,0.70,0.1209,0.0608,0.0223,21.59,15116.0,6
9,0.75,0.1366,0.0661,0.0236,17.98,13484.0,7


![Cutoff](../reports/figures/cutoff_curve.png)

In [8]:
compare_strategies(p, truth, float(best['cutoff_pd'])).set_index('strategy').T

strategy,"A. Approve all, flat $100","B. Model cutoff, flat $100","C. Model cutoff, risk-based limits",D. Risk-based limits + increases after clean repayment
approval_rate,1.0000,0.5000,0.5000,0.5000
first_advance_co_rate,0.1043,0.0469,0.0457,0.0457
avg_advance,100.0000,100.0000,118.7200,268.5000
originations_per_member,811.8800,945.6800,1126.3400,2507.1400
net_loss_rate_of_originations,0.0398,0.0198,0.0143,0.0171
revenue_per_member,65.5200,75.8000,85.8800,122.6500
loss_per_member,32.3200,18.6900,16.1100,42.9500
contribution_per_member,-3.2300,18.8200,30.9100,36.9100
contribution_per_1000_applicants,-3229.0000,9411.0000,15456.0000,18456.0000
payback_month,NaN,6.0000,5.0000,5.0000


![Strategies](../reports/figures/strategy_comparison.png)

![Pricing](../reports/figures/pricing_sensitivity.png)

## 6. Loss curves

![Vintage](../reports/figures/loss_curves_by_vintage.png)

![Score bands](../reports/figures/loss_curves_by_score_band.png)

Later vintages run slightly worse (built-in macro drift), and risk ranks cleanly across score bands. See `reports/credit_committee_memo.md` for the recommendation.